# 1 Ausschnitte erstellen

Schneidet die grossen Pläne (GeoTIFF) in Ausschnitte von 256 × 256 px für das Training von Stufe 1.

Für jeden Plan:
1. pro Label ein Ausschnitt, in dem der Punkt an einer **zufälligen Stelle** liegt (sonst lernt das Modell «Punkte liegen immer in der Mitte»),
2. zusätzlich **zufällige Ausschnitte** im Digitalisierungsperimeter (Beispiele mit wenigen oder keinen Punkten),
3. Pläne mit 300 oder 400 DPI werden auf **600 DPI** hochgerechnet, damit alle Symbole gleich gross sind.

Ausgabe:
- `data/ausschnitte_256px_600dpi/bilder/<plan>_<nr>.png`: Ausschnitt mit 256 × 256 px
- `data/ausschnitte_256px_600dpi/punkte/<plan>.csv`: eine Zeile pro Punkt und Ausschnitt; ein Ausschnitt ohne Punkt hat eine Zeile mit leerem x und y

Pixelkoordinaten: Ursprung oben links, x nach rechts, y nach unten, Mitte von Pixel i liegt bei i + 0.5.
Die Funktionen zum Schneiden (`fenster_lesen`, `ausschnitt_schneiden`) stehen in `grenzpunkte.py`, weil auch Inferenz und V2 sie brauchen.

In [ ]:
import json
import math
import os
import random

import numpy as np
import pandas as pd
import rasterio

from grenzpunkte import DATA, DATENSATZ, GROESSE, ZIEL_DPI, ausschnitt_schneiden

AUSGABE = DATA  # data/ausschnitte_256px_600dpi
RAND = 16  # Mindestabstand eines Punkts vom Ausschnittrand in Pixeln
ANTEIL_ZUFALL = 0.25  # Anzahl zufälliger Ausschnitte = ANTEIL_ZUFALL * Anzahl Labels
SEED = 42
SPALTEN = ["bild", "plan_id", "split", "typ", "punkt_nr", "a_punkt", "x", "y", "ring_abdeckung"]

## Pläne, Split und Perimeter laden
Verwendet werden nur Pläne mit Status `ok`. Der Split ist nach Plan festgelegt (`split.csv`), der Perimeter ist die Hülle um die digitalisierten Punkte plus 5 m.

In [ ]:
plaene = pd.read_csv(f"{DATENSATZ}/plaene.csv")
plaene = plaene[plaene["status"] == "ok"]
split = pd.read_csv(f"{DATENSATZ}/split.csv").set_index("plan_id")["split"]
with open(f"{DATENSATZ}/perimeter_px.json") as f:
    perimeter = json.load(f)
print(f"{len(plaene)} Pläne:", split[plaene["plan_id"]].value_counts().to_dict())

## Ausschnitte eines Plans
Für jeden Ausschnitt wird zuerst die linke obere Ecke im Scan bestimmt, dann wird er geschnitten und gespeichert. Alle Labels werden in Ausschnittkoordinaten umgerechnet: `(x_scan - spalte) * faktor`.
Jeder Plan hat einen eigenen Zufallsgenerator, damit bei jedem Lauf dieselben Ausschnitte entstehen.

In [ ]:
def plan_verarbeiten(plan, split, huelle):
    """Erstellt alle Ausschnitte eines Plans und gibt die Tabelle der Punkte zurück."""
    plan_id = plan["plan_id"]
    zufall = random.Random(f"{SEED}_{plan_id}")  # eigener Zufallsgenerator pro Plan, bei jedem Lauf gleich
    faktor = ZIEL_DPI / plan["scan_dpi"]  # 1.0, 1.5 oder 2.0
    breite_im_scan = GROESSE / faktor

    labels = pd.read_csv(f"{DATENSATZ}/labels/{plan_id}.csv", dtype={"punkt_nr": str})
    symbol_score = pd.read_csv(f"{DATENSATZ}/symbol_score/{plan_id}.csv", dtype={"punkt_nr": str})
    labels = labels.merge(symbol_score[["punkt_nr", "ring_abdeckung"]], on="punkt_nr", how="left")

    # Linke obere Ecke jedes Ausschnitts im Scan bestimmen
    ausschnitte = []  # Einträge (typ, spalte, zeile)
    for _, label in labels.iterrows():
        # zufällige Lage des Punkts im Ausschnitt (in Ausschnittpixeln), daraus die Ecke im Scan
        x_im_ausschnitt = zufall.uniform(RAND, GROESSE - RAND)
        y_im_ausschnitt = zufall.uniform(RAND, GROESSE - RAND)
        spalte = math.floor(label["x_px"] - x_im_ausschnitt / faktor)
        zeile = math.floor(label["y_px"] - y_im_ausschnitt / faktor)
        ausschnitte.append(("punkt", spalte, zeile))

    # zufällige Ausschnitte innerhalb des Rechtecks um die Digitalisierungshülle
    x_min, y_min = huelle.min(axis=0)
    x_max, y_max = huelle.max(axis=0)
    for _ in range(round(ANTEIL_ZUFALL * len(labels))):
        spalte = math.floor(zufall.uniform(x_min, x_max - breite_im_scan))
        zeile = math.floor(zufall.uniform(y_min, y_max - breite_im_scan))
        ausschnitte.append(("zufall", spalte, zeile))

    # Ausschnitte schneiden, speichern und die enthaltenen Punkte notieren
    zeilen = []
    with rasterio.open(f"{DATENSATZ}/geotiff/{plan_id}.tif") as geotiff:
        for nr, (typ, spalte, zeile) in enumerate(ausschnitte):
            name = f"{plan_id}_{nr:05d}"
            ausschnitt_schneiden(geotiff, spalte, zeile, faktor).save(f"{AUSGABE}/bilder/{name}.png")

            # alle Labels in Ausschnittkoordinaten umrechnen und die behalten, die im Ausschnitt liegen
            x = (labels["x_px"] - spalte) * faktor
            y = (labels["y_px"] - zeile) * faktor
            im_ausschnitt = (x >= 0) & (x < GROESSE) & (y >= 0) & (y < GROESSE)

            if not im_ausschnitt.any():
                zeilen.append({"bild": name, "plan_id": plan_id, "split": split, "typ": typ})
            for i in labels.index[im_ausschnitt]:
                zeilen.append({"bild": name, "plan_id": plan_id, "split": split, "typ": typ,
                               "punkt_nr": labels.at[i, "punkt_nr"], "a_punkt": labels.at[i, "a_punkt"],
                               "x": round(x[i], 2), "y": round(y[i], 2),
                               "ring_abdeckung": labels.at[i, "ring_abdeckung"]})

    # Int64 erlaubt leere Werte, ohne dass aus 0 und 1 Kommazahlen werden
    return pd.DataFrame(zeilen, columns=SPALTEN).astype({"a_punkt": "Int64"})

## Alle Pläne verarbeiten
Schreibt rund 40'000 PNG-Dateien und eine Punkttabelle pro Plan. Dauert einige Minuten.

In [ ]:
os.makedirs(f"{AUSGABE}/bilder", exist_ok=True)
os.makedirs(f"{AUSGABE}/punkte", exist_ok=True)

for _, plan in plaene.iterrows():
    plan_id = plan["plan_id"]
    huelle = np.array(perimeter[plan_id]["digitalisierungs_huelle_5m"])
    punkte = plan_verarbeiten(plan, split[plan_id], huelle)
    punkte.to_csv(f"{AUSGABE}/punkte/{plan_id}.csv", index=False)
    print(f"{plan_id}: {punkte['bild'].nunique()} Ausschnitte")

## Kontrolle
Anzahl Ausschnitte pro Split und Art (`punkt` = um ein Label gelegt, `zufall` = zufällig im Perimeter).

In [ ]:
alle = pd.concat(pd.read_csv(f"{AUSGABE}/punkte/{p}.csv") for p in plaene["plan_id"])
pd.crosstab(alle.drop_duplicates("bild")["split"], alle.drop_duplicates("bild")["typ"], margins=True)